# Single-Task Clarity Prediction - Max Pooling with 7-Fold CV

This notebook trains a single-task model for **Clarity classification only**.
Uses **Max Pooling** with sliding window chunking (stride=256) and 7-fold stratified cross-validation.

In [ ]:
%pip install -U -q transformers datasets accelerate scikit-learn "protobuf==3.20.3" sentencepiece matplotlib seaborn

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import os
from collections import Counter
from datasets import load_dataset, Dataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModel,
    TrainingArguments,
    Trainer,
    TrainerCallback
)
from transformers.modeling_outputs import ModelOutput
from transformers.models.roberta.modeling_roberta import RobertaPreTrainedModel, RobertaModel
from dataclasses import dataclass
from typing import Optional, Tuple, Any, Dict, List
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = 'cuda'

## Configuration

In [ ]:
MODEL_NAME = "roberta-large"
MAX_LENGTH = 512
STRIDE = 256
N_FOLDS = 7

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

clarity_label2id = {"Clear Reply": 0, "Ambivalent": 1, "Clear Non-Reply": 2}
clarity_id2label = {v: k for k, v in clarity_label2id.items()}
clarity_labels = ["Clear Reply", "Ambivalent", "Clear Non-Reply"]

## Load and Prepare Data

In [ ]:
print("Loading QEvasion dataset...")
ds = load_dataset("ailsntua/QEvasion")
train_df = ds['train'].to_pandas()
fake_test_df = ds['test'].to_pandas()

print(f"Original train size: {len(train_df)}")

# Add Clear Reply and Clear Non-Reply from test split to combat class imbalance
fake_test_filtered = fake_test_df[fake_test_df['clarity_label'].isin(['Clear Reply', 'Clear Non-Reply'])].copy()
print(f"Fake test filtered size (Clear Reply + Clear Non-Reply only): {len(fake_test_filtered)}")

# Keep only columns that exist in train_df
common_cols = [c for c in train_df.columns if c in fake_test_filtered.columns]
fake_test_filtered = fake_test_filtered[common_cols]

# Combine train with filtered fake test
train_full_df = pd.concat([train_df[common_cols], fake_test_filtered], ignore_index=True)
print(f"Combined train size: {len(train_full_df)}")
print(f"\nClarity label distribution in combined train:")
print(train_full_df['clarity_label'].value_counts())

## Tokenization

In [ ]:
def tokenize_function_train(examples):
    inputs = [
        f"Question: {q}\nAnswer: {a}"
        for q, a in zip(examples["question"], examples["interview_answer"])
    ]
    tokenized = tokenizer(
        inputs,
        truncation=False,
        padding=False,
        max_length=None,
    )
    
    tokenized["labels"] = [clarity_label2id[l] for l in examples["clarity_label"]]
    
    return tokenized

# Tokenize training data
train_full_dataset = Dataset.from_pandas(train_full_df, preserve_index=False)
train_tokenized_full = train_full_dataset.map(
    tokenize_function_train,
    batched=True,
    remove_columns=train_full_dataset.column_names
)

print(f"Train tokenized: {len(train_tokenized_full)} samples")

## Data Collator

In [ ]:
@dataclass
class ClarityDataCollator:
    tokenizer: Any
    
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        labels = torch.tensor(
            [f.pop("labels") for f in features],
            dtype=torch.long
        )
        
        batch = self.tokenizer.pad(
            features,
            padding=True,
            return_tensors="pt"
        )
        batch["labels"] = labels
        
        return batch

data_collator = ClarityDataCollator(tokenizer=tokenizer)

## Model Architecture: Max Pooling for Clarity

In [ ]:
@dataclass
class ClarityClassifierOutput(ModelOutput):
    loss: Optional[torch.FloatTensor] = None
    logits: torch.FloatTensor = None
    hidden_states: Optional[Tuple[torch.FloatTensor, ...]] = None
    attentions: Optional[Tuple[torch.FloatTensor, ...]] = None


class ClarityMaxPooling(RobertaPreTrainedModel):
    
    @classmethod
    def _can_set_experts_implementation(cls) -> bool:
        return False
    
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = 3
        self.config = config

        self.roberta = RobertaModel(config)

        classifier_dropout = (
            config.classifier_dropout
            if hasattr(config, 'classifier_dropout') and config.classifier_dropout is not None
            else config.hidden_dropout_prob
        )
        self.dropout = nn.Dropout(classifier_dropout)

        self.classifier = nn.Linear(config.hidden_size, self.num_labels)

        self.post_init()
    
    def create_chunks_batched(self, input_ids, attention_mask, max_length=512, stride=256):
        batch_size, seq_len = input_ids.shape
        
        all_chunk_ids = []
        all_chunk_masks = []
        chunk_to_sample = []
        
        for sample_idx in range(batch_size):
            sample_input_ids = input_ids[sample_idx]
            sample_attention_mask = attention_mask[sample_idx]
            actual_length = sample_attention_mask.sum().item()

            if actual_length <= max_length:
                chunk_ids = sample_input_ids[:max_length]
                chunk_mask = sample_attention_mask[:max_length]
                
                if len(chunk_ids) < max_length:
                    padding_length = max_length - len(chunk_ids)
                    chunk_ids = torch.cat([
                        chunk_ids,
                        torch.zeros(padding_length, dtype=torch.long, device=input_ids.device)
                    ])
                    chunk_mask = torch.cat([
                        chunk_mask,
                        torch.zeros(padding_length, dtype=torch.long, device=attention_mask.device)
                    ])
                
                all_chunk_ids.append(chunk_ids)
                all_chunk_masks.append(chunk_mask)
                chunk_to_sample.append(sample_idx)
            else:
                start = 0
                while start < actual_length:
                    end = min(start + max_length, actual_length)
                    
                    chunk_ids = sample_input_ids[start:end]
                    chunk_mask = sample_attention_mask[start:end]
                    
                    if len(chunk_ids) < max_length:
                        padding_length = max_length - len(chunk_ids)
                        chunk_ids = torch.cat([
                            chunk_ids,
                            torch.zeros(padding_length, dtype=torch.long, device=input_ids.device)
                        ])
                        chunk_mask = torch.cat([
                            chunk_mask,
                            torch.zeros(padding_length, dtype=torch.long, device=attention_mask.device)
                        ])
                    
                    all_chunk_ids.append(chunk_ids)
                    all_chunk_masks.append(chunk_mask)
                    chunk_to_sample.append(sample_idx)
  
                    start += stride
                    if end >= actual_length:
                        break

        all_chunk_ids = torch.stack(all_chunk_ids, dim=0)
        all_chunk_masks = torch.stack(all_chunk_masks, dim=0)
        chunk_to_sample = torch.tensor(chunk_to_sample, dtype=torch.long, device=input_ids.device)
        
        return all_chunk_ids, all_chunk_masks, chunk_to_sample
    
    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        labels=None,
        output_attentions=None,
        output_hidden_states=None,
        return_dict=None,
    ) -> ClarityClassifierOutput:
        
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict
        batch_size = input_ids.shape[0]

        all_chunk_ids, all_chunk_masks, chunk_to_sample = self.create_chunks_batched(
            input_ids, attention_mask, max_length=512, stride=256
        )

        outputs = self.roberta(
            input_ids=all_chunk_ids,
            attention_mask=all_chunk_masks,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=True,
        )

        all_cls_embeddings = outputs.last_hidden_state[:, 0, :]
        batch_embeddings = []
        for sample_idx in range(batch_size):
            sample_mask = chunk_to_sample == sample_idx
            sample_chunk_embeddings = all_cls_embeddings[sample_mask]
            
            # MAX POOLING
            pooled_embedding = torch.max(sample_chunk_embeddings, dim=0)[0]
            batch_embeddings.append(pooled_embedding)

        pooled_output = torch.stack(batch_embeddings, dim=0)
        pooled_output = self.dropout(pooled_output)

        logits = self.classifier(pooled_output)

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))
        
        return ClarityClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=None,
            attentions=None,
        )

## Metrics & Trainer

In [ ]:
def compute_metrics(eval_pred):
    predictions = eval_pred.predictions
    labels = eval_pred.label_ids

    if hasattr(predictions, 'cpu'):
        predictions = predictions.cpu().numpy()
    if hasattr(labels, 'cpu'):
        labels = labels.cpu().numpy()
    
    preds = np.argmax(predictions, axis=-1)

    accuracy = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='macro')
    
    return {
        'accuracy': accuracy,
        'f1': f1,
    }


class EarlyStoppingCallback(TrainerCallback):
    def __init__(self, patience: int = 3, metric_name: str = "eval_f1", greater_is_better: bool = True):
        self.patience = patience
        self.metric_name = metric_name
        self.greater_is_better = greater_is_better
        self.best_metric = None
        self.patience_counter = 0
        
    def on_evaluate(self, args, state, control, metrics, **kwargs):
        current_metric = metrics.get(self.metric_name)
        
        if current_metric is None:
            return
        
        if self.best_metric is None:
            self.best_metric = current_metric
            self.patience_counter = 0
        else:
            if self.greater_is_better:
                improved = current_metric > self.best_metric
            else:
                improved = current_metric < self.best_metric
            
            if improved:
                self.best_metric = current_metric
                self.patience_counter = 0
            else:
                self.patience_counter += 1
                
        if self.patience_counter >= self.patience:
            print(f"\nEarly stopping triggered!")
            print(f"Best {self.metric_name}: {self.best_metric:.4f}")
            control.should_training_stop = True

## 7-Fold Stratified Cross-Validation

In [ ]:
os.makedirs("./confusion_matrices", exist_ok=True)

print("="*60)
print("7-FOLD STRATIFIED CROSS-VALIDATION - CLARITY ONLY")
print("="*60)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# Storage for out-of-fold predictions
oof_preds = np.zeros(len(train_full_df), dtype=int)
oof_probs = np.zeros((len(train_full_df), 3))
oof_true = np.zeros(len(train_full_df), dtype=int)

# Storage for fold metrics
fold_metrics = []

# Track best model
best_fold = None
best_f1 = 0
best_model_state = None

print(f"\nStarting {N_FOLDS}-fold cross-validation...")
print(f"Total training samples: {len(train_full_df)}\n")

for fold, (train_idx, val_idx) in enumerate(skf.split(train_full_df, train_full_df['clarity_label']), 1):
    print(f"\n{'='*60}")
    print(f"FOLD {fold}/{N_FOLDS}")
    print(f"{'='*60}")
    print(f"Train size: {len(train_idx)}, Val size: {len(val_idx)}")
    
    train_dataset = train_tokenized_full.select(train_idx)
    val_dataset = train_tokenized_full.select(val_idx)
    
    # Initialize model for this fold
    config = AutoConfig.from_pretrained(MODEL_NAME)
    model = ClarityMaxPooling(config)
    
    base_model = AutoModel.from_pretrained(MODEL_NAME)
    model.roberta.load_state_dict(base_model.state_dict())
    del base_model
    
    training_args = TrainingArguments(
        output_dir=f"./results_clarity_maxpool_fold{fold}",
        learning_rate=1e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=15,
        warmup_ratio=0.1,
        weight_decay=0.01,
        max_grad_norm=1.0,
        gradient_checkpointing=True,
        bf16=True,
        bf16_full_eval=True,
        optim="adamw_torch",
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_steps=50,
        seed=SEED + fold,
        report_to="none",
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(patience=3, metric_name="eval_f1")],
    )

    trainer.train()

    val_results = trainer.evaluate()
    fold_metrics.append({
        'fold': fold,
        'val_accuracy': val_results['eval_accuracy'],
        'val_f1': val_results['eval_f1'],
    })
    
    # Track best model
    if val_results['eval_f1'] > best_f1:
        best_f1 = val_results['eval_f1']
        best_fold = fold
        best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    
    print(f"Fold {fold} Results:")
    print(f"  Accuracy: {val_results['eval_accuracy']:.4f}, F1: {val_results['eval_f1']:.4f}")
    
    # OOF predictions
    oof_output = trainer.predict(val_dataset)
    logits_oof = oof_output.predictions

    oof_probs_fold = torch.nn.functional.softmax(torch.tensor(logits_oof), dim=-1).numpy()
    oof_probs[val_idx] = oof_probs_fold
    oof_preds[val_idx] = np.argmax(oof_probs_fold, axis=-1)
    oof_true[val_idx] = [clarity_label2id[label] for label in train_full_df.iloc[val_idx]['clarity_label']]
    
    del model, trainer
    torch.cuda.empty_cache()

print(f"\nBest fold: {best_fold} with F1: {best_f1:.4f}")

## Cross-Validation Results

In [ ]:
print("="*60)
print("CROSS-VALIDATION METRICS")
print("="*60)

metrics_df = pd.DataFrame(fold_metrics)
print(metrics_df.to_string(index=False))

print(f"\nAverage Validation Accuracy: {metrics_df['val_accuracy'].mean():.4f} ± {metrics_df['val_accuracy'].std():.4f}")
print(f"Average Validation Macro F1: {metrics_df['val_f1'].mean():.4f} ± {metrics_df['val_f1'].std():.4f}")

## Out-of-Fold Analysis

In [ ]:
print("="*60)
print("OVERALL OUT-OF-FOLD RESULTS")
print("="*60)

oof_pred_labels = [clarity_id2label[p] for p in oof_preds]
oof_true_labels = [clarity_id2label[t] for t in oof_true]

overall_acc = accuracy_score(oof_true, oof_preds)
overall_f1 = f1_score(oof_true, oof_preds, average='macro')

print(f"\nAccuracy: {overall_acc:.4f}")
print(f"Macro F1: {overall_f1:.4f}")

print("\n--- DETAILED CLASSIFICATION REPORT ---")
print(classification_report(oof_true_labels, oof_pred_labels, labels=clarity_labels, zero_division=0))

## Confusion Matrix

In [ ]:
def plot_confusion_matrix(cm, labels, title, filename, normalize=False):
    """Plot and save confusion matrix"""
    if normalize:
        cm_display = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        fmt = '.3f'
        cmap = 'Blues'
    else:
        cm_display = cm
        fmt = 'd'
        cmap = 'Blues'
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm_display, annot=True, fmt=fmt, cmap=cmap,
                xticklabels=labels, yticklabels=labels,
                cbar_kws={'label': 'Normalized Count' if normalize else 'Count'})
    plt.title(title, fontsize=14, fontweight='bold')
    plt.ylabel('True Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"Saved: {filename}")
    plt.show()

def cm_to_latex(cm, labels, title):
    """Convert confusion matrix to LaTeX table format"""
    n = len(labels)
    
    latex = "\\begin{table}[h]\n"
    latex += "\\centering\n"
    latex += f"\\caption{{{title}}}\n"
    latex += "\\begin{tabular}{l|" + "c" * n + "}\n"
    latex += "\\hline\n"
    
    latex += " & " + " & ".join([f"\\textbf{{{label}}}" for label in labels]) + " \\\\\n"
    latex += "\\hline\n"
    
    for i, label in enumerate(labels):
        row = f"\\textbf{{{label}}}"
        for j in range(n):
            row += f" & {cm[i, j]}"
        row += " \\\\\n"
        latex += row
    
    latex += "\\hline\n"
    latex += "\\end{tabular}\n"
    latex += "\\end{table}\n"
    
    return latex

In [ ]:
print("="*60)
print("CLARITY CONFUSION MATRIX (OUT-OF-FOLD)")
print("="*60)

cm = confusion_matrix(oof_true_labels, oof_pred_labels, labels=clarity_labels)

print("\n--- Raw Confusion Matrix ---")
print(cm)

# Plot raw CM
plot_confusion_matrix(
    cm,
    clarity_labels,
    "Clarity Confusion Matrix (Raw Counts)",
    "./confusion_matrices/clarity_only_raw.png",
    normalize=False
)

# Plot normalized CM
plot_confusion_matrix(
    cm,
    clarity_labels,
    "Clarity Confusion Matrix (Normalized)",
    "./confusion_matrices/clarity_only_normalized.png",
    normalize=True
)

# LaTeX table
print("\n--- LaTeX Table (Raw) ---")
latex_cm = cm_to_latex(cm, clarity_labels, "Clarity Confusion Matrix")
print(latex_cm)

## Per-Fold Statistics

In [ ]:
print("="*60)
print("FOLD-WISE RESULTS TABLE")
print("="*60)

print(f"\n{'Fold':<6} {'Accuracy':>12} {'F1 Score':>12}")
print("-" * 32)
for metric in fold_metrics:
    print(f"{metric['fold']:<6} {metric['val_accuracy']:>12.4f} {metric['val_f1']:>12.4f}")
print("-" * 32)
print(f"{'Mean':<6} {metrics_df['val_accuracy'].mean():>12.4f} {metrics_df['val_f1'].mean():>12.4f}")
print(f"{'Std':<6} {metrics_df['val_accuracy'].std():>12.4f} {metrics_df['val_f1'].std():>12.4f}")

# LaTeX table
print("\n" + "="*60)
print("LATEX TABLE FOR PAPER")
print("="*60)

latex_results = """
\\begin{table}[h]
\\centering
\\caption{7-Fold Cross-Validation Results - Clarity Only (Max Pooling)}
\\begin{tabular}{lcc}
\\hline
\\textbf{Fold} & \\textbf{Accuracy} & \\textbf{Macro F1} \\\\
\\hline
"""

for metric in fold_metrics:
    latex_results += f"Fold {metric['fold']} & {metric['val_accuracy']:.4f} & {metric['val_f1']:.4f} \\\\\n"

latex_results += "\\hline\n"
latex_results += f"Mean & {metrics_df['val_accuracy'].mean():.4f} & {metrics_df['val_f1'].mean():.4f} \\\\\n"
latex_results += f"Std & {metrics_df['val_accuracy'].std():.4f} & {metrics_df['val_f1'].std():.4f} \\\\\n"
latex_results += """\\hline
\\end{tabular}
\\end{table}
"""

print(latex_results)